# Ensemble Model Training

## Overview

This notebook trains multiple models for ensemble prediction:
1. **T5-small**: Transformer-based neural machine translation
2. **mT5-small**: Multilingual T5 (better for low-resource languages)

The ensemble combines strengths of different architectures.

Note: Uses augmented training data (~28,000 samples from original + Akkademia)

In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    T5Tokenizer, T5ForConditionalGeneration,
    MT5Tokenizer, MT5ForConditionalGeneration
)
from sklearn.model_selection import train_test_split
import os
import uuid

os.environ["TOKENIZERS_PARALLELISM"] = "false"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

print("\nLoading augmented data...")
train_df = pd.read_csv('train_augmented.csv')
test_df = pd.read_csv('test.csv')

print(f"Augmented train size: {len(train_df)}")
print(f"Test size: {len(test_df)}")
print("\n--- Data Sources ---")
print(train_df['source'].value_counts())

In [ ]:
# Clean data
train_df = train_df.dropna(subset=['transliteration', 'translation'])
print(f"After dropping NA: {len(train_df)}")

# Split data
train_texts = train_df['transliteration'].tolist()
train_labels = train_df['translation'].tolist()

train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_texts, train_labels, test_size=0.05, random_state=42
)

print(f"Training samples: {len(train_texts)}")
print(f"Validation samples: {len(val_texts)}")

## 1. Dataset Class

In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = "translate Akkadian to English: " + str(self.texts[idx])
        label = str(self.labels[idx])
        
        source = self.tokenizer(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        target = self.tokenizer(
            label,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': source['input_ids'].squeeze(),
            'attention_mask': source['attention_mask'].squeeze(),
            'labels': target['input_ids'].squeeze()
        }

print("Dataset class defined.")

## 2. Train Model 1: T5-small

In [ ]:
from transformers import Trainer, TrainingArguments

print("=" * 60)
print("TRAINING MODEL 1: T5-small")
print("=" * 60)

tokenizer_t5 = T5Tokenizer.from_pretrained("t5-small")
model_t5 = T5ForConditionalGeneration.from_pretrained("t5-small")
model_t5 = model_t5.to(DEVICE)

train_dataset_t5 = TranslationDataset(train_texts, train_labels, tokenizer_t5, max_len=128)
val_dataset_t5 = TranslationDataset(val_texts, val_labels, tokenizer_t5, max_len=128)

training_args_t5 = TrainingArguments(
    output_dir="./ensemble_results/t5_small",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=50,
    weight_decay=0.01,
    logging_steps=200,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    load_best_model_at_end=True,
    report_to="none",
    learning_rate=3e-4,
    save_total_limit=1,
    fp16=torch.cuda.is_available(),
)

trainer_t5 = Trainer(
    model=model_t5,
    args=training_args_t5,
    train_dataset=train_dataset_t5,
    eval_dataset=val_dataset_t5,
)

print("Training T5-small...")
trainer_t5.train()
print("T5-small training completed!")

# Save model
model_t5.save_pretrained("./ensemble_models/t5_small")
tokenizer_t5.save_pretrained("./ensemble_models/t5_small")
print("T5-small model saved.")

## 3. Train Model 2: mT5-small (Optional - skip if time constrained)

In [ ]:
print("=" * 60)
print("TRAINING MODEL 2: mT5-small")
print("=" * 60)

try:
    tokenizer_mt5 = MT5Tokenizer.from_pretrained("google/mt5-small")
    model_mt5 = MT5ForConditionalGeneration.from_pretrained("google/mt5-small")
    model_mt5 = model_mt5.to(DEVICE)
    
    train_dataset_mt5 = TranslationDataset(train_texts, train_labels, tokenizer_mt5, max_len=128)
    val_dataset_mt5 = TranslationDataset(val_texts, val_labels, tokenizer_mt5, max_len=128)
    
    training_args_mt5 = TrainingArguments(
        output_dir="./ensemble_results/mt5_small",
        num_train_epochs=2,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        warmup_steps=50,
        weight_decay=0.01,
        logging_steps=200,
        eval_strategy="steps",
        eval_steps=500,
        save_strategy="steps",
        save_steps=500,
        load_best_model_at_end=True,
        report_to="none",
        learning_rate=3e-4,
        save_total_limit=1,
        fp16=torch.cuda.is_available(),
    )
    
    trainer_mt5 = Trainer(
        model=model_mt5,
        args=training_args_mt5,
        train_dataset=train_dataset_mt5,
        eval_dataset=val_dataset_mt5,
    )
    
    print("Training mT5-small...")
    trainer_mt5.train()
    print("mT5-small training completed!")
    
    # Save model
    model_mt5.save_pretrained("./ensemble_models/mt5_small")
    tokenizer_mt5.save_pretrained("./ensemble_models/mt5_small")
    print("mT5-small model saved.")
    
    HAS_MT5 = True
except Exception as e:
    print(f"mT5 training failed: {e}")
    HAS_MT5 = False

## 4. Save Ensemble Configuration

In [ ]:
# Save ensemble configuration
ensemble_config = {
    'models': ['t5_small'],
    'device': str(DEVICE),
    'training_samples': len(train_texts),
    'val_samples': len(val_texts)
}

if HAS_MT5:
    ensemble_config['models'].append('mt5_small')

import json
with open('ensemble_config.json', 'w') as f:
    json.dump(ensemble_config, f, indent=2)

print("Ensemble configuration saved.")
print(f"Models in ensemble: {ensemble_config['models']}")